In [1]:
import torch
import numpy as np
import copy

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, Dropout
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"O processamento será feito em {device}")

O processamento será feito em cuda


In [3]:
# Carregamento Dados
dataset = load_diabetes()

X_treino, X_teste, y_treino, y_teste = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=42)

# Ajuste dimensões
y_treino = np.expand_dims(y_treino, axis = 1)
y_teste = np.expand_dims(y_teste, axis = 1)

In [4]:
# Normalização alvo: melhora estabilidade gradientes na perda MSE
scaler_y = StandardScaler()
y_treino_scaled = scaler_y.fit_transform(y_treino)
y_teste_scaled = scaler_y.transform(y_teste) # sem fit para não vazar dados

In [5]:
# DataLoaders
batch_size = 8
train_dataset = TensorDataset(Tensor(X_treino), (Tensor(y_treino_scaled)))
test_dataset = TensorDataset(Tensor(X_teste), Tensor(y_teste_scaled))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) 

In [6]:
# Rede + Regularização (Dropout)
model = Sequential(
    Linear(10, 32),
    ReLU(),
    Dropout(0.2),
    Linear(32,16),
    ReLU(),
    Linear(16,1),
).to(device)

In [7]:
# model = Sequential(
#     Linear(10, 16), # classe que instancia a camada de neurônios
#     Dropout(0.2),
#     ReLU(), # função de ativação
#     Linear(16,8),
#     ReLU(),
#     Linear(8,1),
    
# ).to(device)

In [8]:
# Hiperparâmetros Avançados
max_epochs = 500

# weight_decay: penaliza L2 nos pesos (evita overfitting)
optimizer = Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = MSELoss()

# Scheduler: Reduz LR pela metade(factor=0.5) se a perda não melhorar por 15 epochs consecutivas(patience=15)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15)

In [9]:
total_batchs = len(X_treino) // batch_size
last_batch = len(X_treino) % batch_size

print(total_batchs)
print(last_batch)

44
1


In [10]:
def train_and_eval_model():
    best_loss = float('inf')
    patience_counter = 0
    patience_limit = 40 # early stopping
    best_model_weights = copy.deepcopy(model.state_dict())

    for epoch in range(max_epochs):
        # -- FASE DE TREINO --
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

        # -- FASE DE VALIDAÇÃO -- (controle early stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                output = model(batch_x)
                loss = criterion(output, batch_y)
                val_loss = val_loss + loss.item() * batch_x.size(0)

        # Média da perda no dataset de validação
        val_loss = val_loss / len(test_loader.dataset)

        # Atualiza o Scheduler
        scheduler.step(val_loss)

        # Check Early Stopping
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter = patience_counter + 1

        if patience_counter >= patience_limit:
            print(f"Early Stopping acionado! Parando na época {epoch + 1}")
            break
    # No fim (ou interrupção do loop), restaura melhores pesos encontrados
    model.load_state_dict(best_model_weights)
    print("Treinamento finalizado. Modelo revertido para melhor época validada.")


def eval_model():
    model.eval()
    with torch.no_grad():
        X_teste_tensor = Tensor(X_teste).to(device)
        
        # O modelo prevê na escala normalizada
        output_scaled_tensor = model(X_teste_tensor)
        output_scaled = output_scaled_tensor.cpu().numpy()

        # REVERTER A NORMALIZAÇÃO para poder calcular o erro real (MSE, RMSE nas unidades originais)
        output_real = scaler_y.inverse_transform(output_scaled)

        print(f"R^2: {r2_score(y_teste, output_real)}")
        print(f"RMSE: {root_mean_squared_error(y_teste, output_real)}")
        print(f"MSE: {mean_squared_error(y_teste, output_real)}")


In [11]:
# Executar as funções

train_and_eval_model()
eval_model()

Early Stopping acionado! Parando na época 77
Treinamento finalizado. Modelo revertido para melhor época validada.
R^2: 0.5294585192515181
RMSE: 49.92995262342195
MSE: 2493.0001689771602


como funciona quando fazem distilling ou scraping? Eles associam a resposta que o modelo fez com os possíveis pesos que seria de output?